# fastmcp_slim smoke test

Tests basic import + primary API surface of FastMCP (slim variant). FastMCP is a high-level framework for building MCP (Model Context Protocol) servers and clients.

In [ ]:
# Example 1: Construct a FastMCP server instance and inspect its identity.
from fastmcp import FastMCP

mcp = FastMCP(name="DemoServer")

print(f"server type: {type(mcp).__name__}")
print(f"server name: {mcp.name}")

assert isinstance(mcp, FastMCP)
assert mcp.name == "DemoServer"

In [ ]:
# Example 2: Register a tool with the @tool decorator and verify it is tracked.
# fastmcp 3.x: introspection is list_tools() / get_tool(name) (get_tools() removed).
from fastmcp import FastMCP

mcp = FastMCP(name="ToolServer")

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

@mcp.tool
def greet(name: str) -> str:
    """Return a greeting."""
    return f"Hello, {name}!"

tool_names = sorted(t.name for t in await mcp.list_tools())
print(f"registered tools: {tool_names}")
assert "add" in tool_names
assert "greet" in tool_names

add_tool = await mcp.get_tool("add")
print(f"get_tool('add') -> {type(add_tool).__name__}")
assert add_tool.name == "add"


In [ ]:
# Example 3: Register a resource and a resource template.
# fastmcp 3.x: list_resources() / list_resource_templates() (get_* dicts removed).
from fastmcp import FastMCP

mcp = FastMCP(name="ResourceServer")

@mcp.resource("config://app/name")
def app_name() -> str:
    return "fastmcp-demo"

@mcp.resource("users://{user_id}/profile")
def user_profile(user_id: str) -> dict:
    return {"id": user_id, "name": f"User {user_id}"}

resource_uris = [str(r.uri) for r in await mcp.list_resources()]
templates = await mcp.list_resource_templates()
print(f"resources: {resource_uris}")
print(f"templates: {[t.uri_template for t in templates]}")

assert any("config://app/name" in u for u in resource_uris)
assert any(t.uri_template == "users://{user_id}/profile" for t in templates)


In [ ]:
# Example 4: Register a prompt and verify discovery.
# fastmcp 3.x: list_prompts() (get_prompts() dict removed).
from fastmcp import FastMCP

mcp = FastMCP(name="PromptServer")

@mcp.prompt
def review_code(code: str) -> str:
    """Generate a review request for code."""
    return f"Please review:\n\n{code}"

@mcp.prompt
def summarize(text: str) -> str:
    return f"Summarize: {text}"

prompt_names = sorted(p.name for p in await mcp.list_prompts())
print(f"registered prompts: {prompt_names}")

assert "review_code" in prompt_names
assert "summarize" in prompt_names


In [ ]:
# Example 5: In-memory client-server round trip via FastMCP Client.
from fastmcp import Client, FastMCP

server = FastMCP(name="EchoServer")

@server.tool
def echo(message: str) -> str:
    return f"echo: {message}"

async with Client(server) as client:
    tools = await client.list_tools()
    result = await client.call_tool("echo", {"message": "hi"})

tool_names = [t.name for t in tools]
print(f"client saw tools: {tool_names}")
print(f"call result: {result.data}")

assert "echo" in tool_names
assert "hi" in str(result.data)